# 01 – EC Antitrust Master List

## Ziel dieses Notebooks

Dieses Notebook liest eine lokale JSON-Datei mit EC-Wettbewerbsfällen ein, filtert daraus ausschliesslich **Antitrust-Fälle** heraus und exportiert eine saubere, flache Mastertabelle.

### Warum ist die EC-Masterliste der erste Schritt?

Die Europäische Kommission (EC) ist die zentrale Behörde für Wettbewerbsrecht in der EU. Ihre Falldatenbank ist der Ausgangspunkt für alle weiteren Analysen:
- Ohne eine saubere EC-Liste können wir keine CJEU-Urteile zuordnen.
- Ohne klare Fall-IDs können wir keine Verknüpfungen herstellen.
- Ohne Filterung auf Antitrust würden Merger-, Beihilfe- und DMA-Fälle die Analyse verfälschen.



## 1 – Imports und Konfiguration


In [1]:
import json
from pathlib import Path

import pandas as pd

# Pandas-Ausgabe etwas breiter stellen, damit Spalten lesbar bleiben
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

print("Imports OK")


Imports OK


## 2 – Input- und Output-Pfade definieren

Hier legen wir fest, wo die Quelldatei liegt und wohin die Ergebnisse gespeichert werden.
Alle Pfade werden mit `pathlib.Path` definiert – das funktioniert auf Windows, Mac und Linux gleich.


In [2]:
# Pfad zur Quelldatei (JSON oder Excel)
# Source von: https://data.europa.eu/data/datasets/18489cb7-bce7-4d44-a138-795b390d2109~~1?locale=en
# --> https://compcases-open-data-portal-files-prod.s3.eu-west-1.amazonaws.com/case-data-AT.json

INPUT_FILE = Path("data/case-data-AT.json")

# Ausgabeordner – wird automatisch erstellt, falls er nicht existiert
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ausgabedateien
OUTPUT_CSV     = OUTPUT_DIR / "ec_antitrust_master.csv"

print(f"Input : {INPUT_FILE.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")
print(f"Datei existiert: {INPUT_FILE.exists()}")


Input : C:\vsCodeWorkspace\masterZHAW\masterarbeit\data_analysis_master\data\case-data-AT.json
Output: C:\vsCodeWorkspace\masterZHAW\masterarbeit\data_analysis_master\data\processed
Datei existiert: True


## 3 – Daten laden

Die Quelldatei ist eine JSON-Datei, in der jeder Schlüssel eine EC-Fall-ID ist (z. B. `AT.39294`).
Der Wert ist ein Objekt mit `metadata`, `caseAttachments` und `decisions`.


In [3]:
def load_input_file(path: Path):
    """Lädt JSON – gibt ein dict (JSON) zurück."""
    suffix = path.suffix.lower()
    if suffix == ".json":
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        print(f"JSON geladen: {len(data)} Einträge (Top-Level-Keys)")
        return "json", data
    else:
        raise ValueError(f"Unbekanntes Dateiformat: {suffix}. Erwartet: .json")


file_type, raw_data = load_input_file(INPUT_FILE)


JSON geladen: 748 Einträge (Top-Level-Keys)


## 4 – Hilfsfunktionen

Die JSON-Felder sind fast immer **Listen** (auch wenn nur ein Wert drin ist).
Ausserdem sind manche Felder wie `caseSectors` oder `caseLegalBasis` als **JSON-Strings innerhalb der Liste** kodiert.

Diese kleinen Hilfsfunktionen machen den Code übersichtlich und robust.


In [4]:
def first_value(x):
    """Gibt den ersten Eintrag einer Liste zurück, oder None wenn leer/kein Wert."""
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return None


print("Hilfsfunktionen definiert.")

# Kurze Tests
assert first_value(["a", "b"]) == "a"
assert first_value([]) is None
print("Alle Tests bestanden.")


Hilfsfunktionen definiert.
Alle Tests bestanden.


## 5 – Antitrust-Filterlogik

### Warum filtern wir?

Die EC-Datenbank enthält verschiedene Falltypen:
- **Antitrust** (Art. 101/102 TFEU) – das wollen wir
- **Cartels** – oft Teil von Antitrust, aber separat klassifiziert
- **Mergers** – Fusionskontrolle, nicht relevant
- **State Aid** – Beihilferecht, nicht relevant
- **DMA / FSR** – neuere Instrumente, nicht relevant

### Filterkriterien

Wir behalten einen Fall, wenn **mindestens eines** dieser Kriterien zutrifft:
1. `caseCartel` enthält `"Antitrust"` oder `"Cartel"` → primäres Kriterium
2. `caseInstrument` enthält `"Antitrust"` → sekundäres Kriterium

Fälle mit `caseInstrument` wie `"Merger"`, `"State Aid"`, `"DMA"`, `"FSR"` werden **ausgeschlossen**.


In [5]:
# Schlüsselwörter, die auf Antitrust hinweisen
ANTITRUST_KEYWORDS = {"antitrust", "cartel"}

# Schlüsselwörter, die explizit NICHT Antitrust sind
EXCLUDE_KEYWORDS = {"merger", "state aid", "dma", "fsr"}


def is_antitrust_case(metadata: dict) -> bool:
    """
    Entscheidet, ob ein Fall ein Antitrust-Fall ist.

    Gibt True zurück, wenn:
    - caseCartel oder caseInstrument einen Antitrust-Begriff enthält
    UND
    - kein Ausschluss-Begriff (Merger, State Aid, etc.) vorkommt
    """
    cartel_val = " ".join(str(v) for v in metadata.get("caseCartel", []) if v).lower()
    instr_val  = " ".join(str(v) for v in metadata.get("caseInstrument", []) if v).lower()
    combined   = cartel_val + " " + instr_val

    has_antitrust = any(kw in combined for kw in ANTITRUST_KEYWORDS)
    has_exclusion = any(kw in combined for kw in EXCLUDE_KEYWORDS)

    return has_antitrust and not has_exclusion


print("Filterlogik definiert.")

# Kurze Tests
assert is_antitrust_case({"caseCartel": ["Antitrust"], "caseInstrument": ["Antitrust & Cartels"]}) == True
assert is_antitrust_case({"caseCartel": ["Cartel"],   "caseInstrument": ["Antitrust & Cartels"]}) == True
assert is_antitrust_case({"caseCartel": [],            "caseInstrument": ["Merger"]})              == False
assert is_antitrust_case({"caseCartel": ["Antitrust"], "caseInstrument": ["Merger"]})              == False
print("Alle Filter-Tests bestanden.")


Filterlogik definiert.
Alle Filter-Tests bestanden.


## 6 – Flattening: Aus JSON-Struktur eine flache Tabelle bauen

Jeder Fall in der JSON-Datei hat eine verschachtelte Struktur.
Wir "flatten" diese Struktur zu einer einzigen Zeile pro Fall.

Die Funktion `extract_case_row()` macht genau das: Sie nimmt einen Fall und gibt ein Dictionary zurück,
das direkt als Zeile in einem DataFrame verwendet werden kann.

### Datums-Fallback-Kette (moderne JSON-Fälle)

Priorität für `date` (erstes nicht-leeres Datum gewinnt):
1. `caseLastDecisionDate`
2. `decisionAdoptionDate` (aus decisions[])
3. `decisionOfficialJournalPublicationsPublishedDates` (aus decisions[])
4. `attachmentDocumentDate` (aus caseAttachments[])
5. `attachmentSentDate` (aus caseAttachments[])
6. `attachmentPublicationBusinessDate` (aus caseAttachments[])
7. `caseOfficialJournalPublicationsPublishedDates`
8. `caseInitiationDate` (letzter Fallback)

Das Feld `date_source` dokumentiert, welche Quelle verwendet wurde.


In [6]:
def get_first_decision_attachment_link(case_obj: dict) -> str | None:
    """Gibt den ersten attachmentLink aus decisions[].decisionAttachments[] zurück, oder None."""
    for decision in case_obj.get("decisions", []):
        for att in decision.get("decisionAttachments", []):
            link = first_value(att.get("metadata", {}).get("attachmentLink", []))
            if link:
                return link
    return None


def get_first_case_attachment_link(case_obj: dict) -> str | None:
    """Gibt den ersten attachmentLink aus caseAttachments[] zurück, oder None."""
    for att in case_obj.get("caseAttachments", []):
        link = first_value(att.get("metadata", {}).get("attachmentLink", []))
        if link:
            return link
    return None


def get_document_type_and_url(case_obj: dict) -> tuple[str, str]:
    """
    Bestimmt document_type und document_url nach Prioritätsregel:
    A. decisionAttachments -> 'decision'
    B. caseAttachments     -> 'case'
    C. sonst               -> 'none', ''
    """
    link = get_first_decision_attachment_link(case_obj)
    if link:
        return "decision", link
    link = get_first_case_attachment_link(case_obj)
    if link:
        return "case", link
    return "none", ""


def _nonempty(val: str | None) -> str | None:
    """Gibt val zurück wenn nicht leer/None, sonst None."""
    if val and str(val).strip():
        return str(val).strip()
    return None


def resolve_date(case_obj: dict) -> tuple[str | None, str]:
    """
    Bestimmt das finale Datum und die Datumsquelle nach der Fallback-Kette.

    Rückgabe: (date_str_or_None, date_source_label)
    """
    meta = case_obj.get("metadata", {})

    # 1. caseLastDecisionDate
    val = _nonempty(first_value(meta.get("caseLastDecisionDate", [])))
    if val:
        return val, "case_last_decision"

    # 2. decisionAdoptionDate (aus decisions[])
    for dec in case_obj.get("decisions", []):
        val = _nonempty(first_value(dec.get("metadata", {}).get("decisionAdoptionDate", [])))
        if val:
            return val, "decision_adoption"

    # 3. decisionOfficialJournalPublicationsPublishedDates (aus decisions[])
    for dec in case_obj.get("decisions", []):
        val = _nonempty(first_value(dec.get("metadata", {}).get("decisionOfficialJournalPublicationsPublishedDates", [])))
        if val:
            return val, "decision_oj"

    # 4. attachmentDocumentDate (aus caseAttachments[])
    for att in case_obj.get("caseAttachments", []):
        val = _nonempty(first_value(att.get("metadata", {}).get("attachmentDocumentDate", [])))
        if val:
            return val, "attachment_document"

    # 5. attachmentSentDate (aus caseAttachments[])
    for att in case_obj.get("caseAttachments", []):
        val = _nonempty(first_value(att.get("metadata", {}).get("attachmentSentDate", [])))
        if val:
            return val, "attachment_sent"

    # 6. attachmentPublicationBusinessDate (aus caseAttachments[])
    for att in case_obj.get("caseAttachments", []):
        val = _nonempty(first_value(att.get("metadata", {}).get("attachmentPublicationBusinessDate", [])))
        if val:
            return val, "attachment_publication"

    # 7. caseOfficialJournalPublicationsPublishedDates
    val = _nonempty(first_value(meta.get("caseOfficialJournalPublicationsPublishedDates", [])))
    if val:
        return val, "case_oj"

    # 8. caseInitiationDate (letzter Fallback)
    val = _nonempty(first_value(meta.get("caseInitiationDate", [])))
    if val:
        return val, "initiation"

    return None, "missing"


def extract_case_row(case_id: str, case_obj: dict) -> dict:
    """
    Extrahiert die relevanten Felder aus einem Fall-Objekt.

    Parameter:
        case_id  : der Schlüssel aus dem JSON (z. B. 'AT.39294')
        case_obj : das vollständige Fall-Objekt mit metadata, caseAttachments, decisions
    """
    meta = case_obj.get("metadata", {})
    doc_type, doc_url = get_document_type_and_url(case_obj)
    date_val, date_source = resolve_date(case_obj)

    row = {
        "ec_case_number": first_value(meta.get("caseNumber", [])) or case_id,
        "case_title"    : first_value(meta.get("caseTitle", [])),
        "date"          : date_val,
        "date_source"   : date_source,
        "type"          : first_value(meta.get("caseType", [])),
        "document_type" : doc_type,
        "document_url"  : doc_url,
    }
    return row


print("extract_case_row() und Dokument-Hilfsfunktionen definiert.")


extract_case_row() und Dokument-Hilfsfunktionen definiert.


## 7 – Hauptverarbeitung: Alle Fälle durchlaufen, filtern und flattenen

Jetzt kombinieren wir alles:
1. Jeden Fall aus der JSON-Datei lesen
2. Prüfen, ob es ein Antitrust-Fall ist
3. Falls ja: in eine flache Zeile umwandeln
4. Alle Zeilen zu einem DataFrame zusammenführen


In [7]:
source_file_name = INPUT_FILE.name
rows = []
skipped_non_antitrust = 0
skipped_errors = 0

if file_type == "json":
    for case_id, case_obj in raw_data.items():
        try:
            meta = case_obj.get("metadata", {})

            # Antitrust-Filter anwenden
            if not is_antitrust_case(meta):
                skipped_non_antitrust += 1
                continue

            row = extract_case_row(case_id, case_obj)
            rows.append(row)

        except Exception as e:
            print(f"  WARNUNG: Fehler bei Fall '{case_id}': {e}")
            skipped_errors += 1


# DataFrame erstellen
df = pd.DataFrame(rows)

print(f"\nVerarbeitung abgeschlossen:")
print(f"  Antitrust-Fälle gefunden      : {len(df)}")
print(f"  Nicht-Antitrust übersprungen  : {skipped_non_antitrust}")
print(f"  Fehler beim Verarbeiten       : {skipped_errors}")



Verarbeitung abgeschlossen:
  Antitrust-Fälle gefunden      : 748
  Nicht-Antitrust übersprungen  : 0
  Fehler beim Verarbeiten       : 0


## 8 – Normalisierung der Datumsspalten


In [8]:
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
df["date"] = df["date"].where(df["date"].notna(), other=None)
df = df.replace("", None)

print("Datumsspalten normalisiert.")
print(df[["ec_case_number", "date", "date_source"]].head(5))


Datumsspalten normalisiert.
  ec_case_number        date          date_source
0       AT.35803  1996-01-29  attachment_document
1       AT.34950  2001-06-15   case_last_decision
2       AT.39172  2007-01-10   case_last_decision
3       AT.39294  2010-06-25  attachment_document
4       AT.39173  2007-01-10   case_last_decision


## 9 – Qualitätschecks

Bevor wir exportieren, prüfen wir die Qualität der Daten:
- Wie viele Fälle haben wir?
- Wie viele Case IDs fehlen?
- Gibt es Dubletten?
- Wie verteilen sich die Falltypen?


In [9]:
print("=" * 60)
print("QUALITÄTSCHECKS")
print("=" * 60)

total_input = len(raw_data)
print(f"\nAnzahl Input-Fälle (gesamt in Datei) : {total_input}")
print(f"Anzahl Antitrust-Fälle (nach Filter) : {len(df)}")
print(f"Anteil Antitrust                     : {len(df) / total_input * 100:.1f}%")

missing_ids   = df["ec_case_number"].isna().sum()
dupes         = df["ec_case_number"].duplicated().sum()
missing_title = df["case_title"].isna().sum()
missing_date  = df["date"].isna().sum()

print(f"\nFehlende Case IDs (ec_case_number)   : {missing_ids}")
print(f"Dubletten (ec_case_number)           : {dupes}")
print(f"Fehlende Falltitel                   : {missing_title}")
print(f"Fehlende Datumswerte                 : {missing_date}")


QUALITÄTSCHECKS

Anzahl Input-Fälle (gesamt in Datei) : 748
Anzahl Antitrust-Fälle (nach Filter) : 748
Anteil Antitrust                     : 100.0%

Fehlende Case IDs (ec_case_number)   : 0
Dubletten (ec_case_number)           : 0
Fehlende Falltitel                   : 0
Fehlende Datumswerte                 : 0


In [10]:
print("\nHäufigkeitsverteilung: type")
print("-" * 40)
print(df["type"].value_counts(dropna=False).to_string())



Häufigkeitsverteilung: type
----------------------------------------
type
AtStandardATCCase      704
AtStateMeasureCase      38
AtSectorInquiryCase      6


In [11]:
print("\nÜbersicht aller Spalten und Datentypen:")
print(df.dtypes)
print(f"\nShape: {df.shape[0]} Zeilen × {df.shape[1]} Spalten")



Übersicht aller Spalten und Datentypen:
ec_case_number    str
case_title        str
date              str
date_source       str
type              str
document_type     str
document_url      str
dtype: object

Shape: 748 Zeilen × 7 Spalten


## 10 – Export


In [12]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"CSV gespeichert: {OUTPUT_CSV.resolve()}")
print(f"  Dateigrösse: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")


CSV gespeichert: C:\vsCodeWorkspace\masterZHAW\masterarbeit\data_analysis_master\data\processed\ec_antitrust_master.csv
  Dateigrösse: 100.6 KB
